In [ ]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc
from dash import html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from animal_shelter import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = os.getenv("DB_USER")
password = os.getenv("DB_PASS")
host = os.getenv("DB_HOST")
port = int(os.getenv("DB_PORT", 32256))
db = "AAC"
collection = "animals"

# Fallback, keeps your app working if env vars not set
if username is None:
    username = "aacuser"
if password is None:
    password = "6192530Maripositas"
if host is None:
    host = "nv-desktop-services.apporto.com"

db = AnimalShelter(username, password, host, port, db, collection)

# Optimization: Instead of repeatedly processing the full dataset in memory,
# we prepare the data structure once and reuse it for filtering operations.
# This reduces unnecessary repeated computations and improves efficiency.
records = db.read({})

# Safety check to avoid crashes if no data returned
if records:
    df = pd.DataFrame.from_records(records)
else:
    df = pd.DataFrame()

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Load and encode image
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

# Define the app layout
app.layout = html.Div([
    html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()), style={'height': '100px'}),
    html.P("Created by Kim Cortez • June 2025", style={'fontSize': '16px', 'fontStyle': 'italic'}),
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    html.Hr(),
    
    # Dropdown + Table
    html.Div([
        html.Label("Filter by Rescue Type:"),
        dcc.Dropdown(
            id='rescue-type-dropdown',
            options=[
                {'label': 'Water Rescue', 'value': 'Water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'Mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'Disaster'},
                {'label': 'Reset', 'value': 'Reset'}
            ],
            placeholder='Select a rescue type',
            style={'width': '50%'}
        ),
        html.Hr(),
        dash_table.DataTable(
    id='datatable-id',
    columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
    data=df.to_dict('records'),
    row_selectable='single',         
    selected_rows=[],                
    style_table={'overflowX': 'auto'},
    style_cell={'textAlign': 'left'}
),
        html.Br(),
        html.Hr()
    ]),

    # Side-by-side charts
    html.Div(className='row',
        style={'display': 'flex'},
        children=[
            html.Div(id='graph-id', className='col s12 m6'),
            html.Div(id='map-id', className='col s12 m6')
        ]
    )
])
#############################################
# Interaction Between Components / Controller
#############################################

from dash.dependencies import Input, Output

@app.callback(
    Output('datatable-id', 'data'),
    [Input('rescue-type-dropdown', 'value')]
)
def filter_rescue_data(rescue_type):
    rescue_queries = {
        'Water': {
            '$and': [
                {'breed': {'$regex': 'Labrador Retriever|Chesapeake Bay Retriever|Newfoundland', '$options': 'i'}},
                {'sex_upon_outcome': 'Intact Female'},
                {'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}}
            ]
        },
        'Mountain': {
            '$and': [
                {'breed': {'$regex': 'German Shepherd|Alaskan Malamute|Old English Sheepdog|Siberian Husky|Rottweiler', '$options': 'i'}},
                {'sex_upon_outcome': 'Intact Male'},
                {'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}}
            ]
        },
        'Disaster': {
            '$and': [
                {'breed': {'$regex': 'Doberman Pinscher|German Shepherd|Golden Retriever|Bloodhound|Rottweiler', '$options': 'i'}},
                {'sex_upon_outcome': 'Intact Male'},
                {'age_upon_outcome_in_weeks': {'$gte': 20, '$lte': 300}}
            ]
        }
    }

    query = rescue_queries.get(rescue_type, {})
    records = db.read(query)

    if records:
        filtered_df = pd.DataFrame.from_records(records)
        if '_id' in filtered_df.columns:
            filtered_df.drop(columns=['_id'], inplace=True)
    else:
        filtered_df = pd.DataFrame(columns=df.columns)

    return filtered_df.to_dict('records')


@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    if viewData is None or len(viewData) == 0:
        return [html.P("No data to display.")]
    
    dff = pd.DataFrame.from_dict(viewData)
    fig = px.pie(dff, names='breed', title='Breed Distribution in Selected Filter')
    
    return [
        dcc.Graph(figure=fig)
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:
        return []
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [
        Input('datatable-id', "derived_virtual_data"),
        Input('datatable-id', "derived_virtual_selected_rows")
    ]
)
def update_map(viewData, index):  
    if viewData is None or index is None or len(index) == 0:
        return [html.P("Select a row to see its location.", style={'textAlign': 'center'})]
    
    dff = pd.DataFrame(viewData)
    row = index[0]

    # Default to Austin TX if lat/lon are missing
    try:
        lat = dff.iloc[row]["location_lat"]
        lon = dff.iloc[row]["location_long"]
        name = dff.iloc[row]["name"]
        breed = dff.iloc[row]["breed"]
    except Exception as e:
        return [html.P("Location data not available.", style={'textAlign': 'center'})]

    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[lat, lon], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            dl.Marker(position=[lat, lon], children=[
                dl.Tooltip(f"{breed}"),
                dl.Popup([
                    html.H4("Animal Name"),
                    html.P(name)
                ])
            ])
        ])
    ]

app.run_server(debug=True, use_reloader=False)
